# Stage 9: Power BI Aggregate Exports

Stage 9 prepares the aggregate sequence-cluster composition table used for the pre-endpoint activity-history visual in the Power BI artefact.

The export summarises how all six pre-endpoint sequence clusters are distributed within each of the four observed age-18 destinations. Participant identifiers are used only to link the Stage 1 outcome and sequence-cluster data and are not written to the Power BI export.

In [1]:
# Project paths

from pathlib import Path
import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()

project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (
            directory
            / "data_derived"
            / "stage_1_outcome_construction"
            / "age18_activity_outcome.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("The project folder could not be located.")

data_derived = project_root / "data_derived"
stage_1_outcome_dir = data_derived / "stage_1_outcome_construction"
stage_1_sequence_dir = data_derived / "stage_1_supplementary_sequence"
powerbi_dir = data_derived / "stage_9_powerbi_aggregate_exports"
powerbi_dir.mkdir(parents=True, exist_ok=True)

print("Project root located.")
print(
    "Power BI export folder:",
    powerbi_dir.relative_to(project_root).as_posix(),
)

Project root located.
Power BI export folder: data_derived/stage_9_powerbi_aggregate_exports


In [2]:
# Load the inputs used for the Power BI sequence-cluster composition table

outcome_path = stage_1_outcome_dir / "age18_activity_outcome.csv"
sequence_path = stage_1_sequence_dir / "pre_endpoint_activity_sequence_clusters.csv"

required_paths = [outcome_path, sequence_path]
missing_paths = [
    path.relative_to(project_root).as_posix()
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(f"Missing input file(s): {missing_paths}")

outcome_data = pd.read_csv(outcome_path, dtype={"NSID": "string"})
sequence_clusters = pd.read_csv(sequence_path, dtype={"NSID": "string"})

outcome_data["NSID"] = outcome_data["NSID"].str.strip()
sequence_clusters["NSID"] = sequence_clusters["NSID"].str.strip()

print("Age-18 outcome sample:", f"{len(outcome_data):,}")
print("Complete sequence sample:", f"{len(sequence_clusters):,}")

Age-18 outcome sample: 9,767
Complete sequence sample: 9,724


In [3]:
# Create sequence-cluster composition table for Power BI

cluster_powerbi_source = (
    sequence_clusters[["NSID", "sequence_cluster"]]
    .merge(
        outcome_data[["NSID", "age18_outcome"]],
        on="NSID",
        how="inner",
        validate="one_to_one",
    )
)

destination_sizes = (
    cluster_powerbi_source
    .groupby("age18_outcome")
    .size()
)

powerbi_cluster_composition = (
    cluster_powerbi_source
    .groupby(
        ["age18_outcome", "sequence_cluster"],
        observed=True,
    )
    .size()
    .reset_index(name="Count")
)

powerbi_cluster_composition[
    "Percentage within destination"
] = (
    powerbi_cluster_composition["Count"]
    / powerbi_cluster_composition["age18_outcome"].map(destination_sizes)
    * 100
)

print("Rows:", len(powerbi_cluster_composition))
print(
    "Smallest cell count:",
    int(powerbi_cluster_composition["Count"].min()),
)
print(
    "Cells below 10:",
    int((powerbi_cluster_composition["Count"] < 10).sum()),
)

display(
    powerbi_cluster_composition.sort_values(
        ["age18_outcome", "Count"],
        ascending=[True, False],
    ).round(2)
)

Rows: 24
Smallest cell count: 7
Cells below 10: 1


,age18_outcome,sequence_cluster,Count,Percentage within destination
3,Apprenticeship or training,Predominantly apprenticeship/training,264,50.48
4,Apprenticeship or training,Predominantly education,142,27.15
1,Apprenticeship or training,Later transition to employment,39,7.46
0,Apprenticeship or training,Earlier transition to employment,33,6.31
5,Apprenticeship or training,Predominantly employed,29,5.54
2,Apprenticeship or training,Predominantly NEET,16,3.06
10,Education,Predominantly education,4928,96.17
6,Education,Earlier transition to employment,121,2.36
11,Education,Predominantly employed,30,0.59
8,Education,Predominantly NEET,24,0.47


In [4]:
# Validate the final Power BI composition table

public_cluster_composition = powerbi_cluster_composition.copy()

if "NSID" in public_cluster_composition.columns:
    raise ValueError("Participant identifiers must not be included in the Power BI export.")

expected_rows = 4 * 6
if len(public_cluster_composition) != expected_rows:
    raise ValueError(
        f"Expected {expected_rows} destination-cluster rows, found {len(public_cluster_composition)}."
    )

clusters_per_destination = (
    public_cluster_composition.groupby("age18_outcome")["sequence_cluster"]
    .nunique()
)

if not (clusters_per_destination == 6).all():
    raise ValueError("Each age-18 destination must contain all six sequence clusters.")

display(
    public_cluster_composition.sort_values(
        ["age18_outcome", "Count"],
        ascending=[True, False],
    ).round(2)
)

,age18_outcome,sequence_cluster,Count,Percentage within destination
3,Apprenticeship or training,Predominantly apprenticeship/training,264,50.48
4,Apprenticeship or training,Predominantly education,142,27.15
1,Apprenticeship or training,Later transition to employment,39,7.46
0,Apprenticeship or training,Earlier transition to employment,33,6.31
5,Apprenticeship or training,Predominantly employed,29,5.54
2,Apprenticeship or training,Predominantly NEET,16,3.06
10,Education,Predominantly education,4928,96.17
6,Education,Earlier transition to employment,121,2.36
11,Education,Predominantly employed,30,0.59
8,Education,Predominantly NEET,24,0.47


In [5]:
# Save the sequence-cluster composition table for Power BI

export_path = (
    powerbi_dir
    / "powerbi_sequence_cluster_composition.csv"
)

public_cluster_composition.to_csv(
    export_path,
    index=False,
)

print(
    "Saved:",
    export_path.relative_to(project_root).as_posix(),
)
print("Rows:", len(public_cluster_composition))
print(
    "Sequence clusters per destination:",
    public_cluster_composition.groupby("age18_outcome")["sequence_cluster"].nunique().to_dict(),
)

Saved: data_derived/stage_9_powerbi_aggregate_exports/powerbi_sequence_cluster_composition.csv
Rows: 24
Sequence clusters per destination: {'Apprenticeship or training': 6, 'Education': 6, 'Employment': 6, 'Unemployment or inactivity (NEET)': 6}


## Output used in Power BI

`data_derived/stage_9_powerbi_aggregate_exports/powerbi_sequence_cluster_composition.csv`